In [ ]:
# Cell 1 — Run configuration
import time
CONTROLLER_SESSION_START_UNIX=time.time()
RUN_SEED = 42  # CHANGE ONLY THIS: 42 / 1337 / 2026 / 31415 / 271828

PUBLICATION_SEEDS=[42,1337,2026,31415,271828]
if RUN_SEED not in PUBLICATION_SEEDS:
    raise ValueError(f"RUN_SEED must be one of {PUBLICATION_SEEDS}, got {RUN_SEED}")

MODEL_ID="neulab/Pangea-7B-hf"
MODEL_REVISION="9394c5693ad814cea885d21c34c98b71fcd1fc1b"
PANGEA_NATIVE_BINDING_SHA256="c7052cc933ab70161088728ebf82d4944d60ba13f0d2a0d9548a3b7de453d2eb"
SHARED_FORWARD_PROTOCOL_SHA256="37b7d30a994fb98225d49d76cca5768a838d456abf844dc1a5d93019a0cd15aa"
PARENT_GATED_PROTOCOL_SHA256="627bdf5a77ab694581ff590c32f8885c901ebad09e8df86dbab07762caaf95dd"

# Validation-selected Pangea Frozen Common Gated v1.1 states established by the completed five seed runs.
GATED_BEST_STEP_BY_SEED={
    42:800,
    1337:800,
    2026:932,
    31415:932,
    271828:932,
}
GATED_VALIDATION_FREEZE_ID_BY_SEED={
    42:"059dac5a0b4362860ee5f6c6ec66200a34eb6c50f0a8d35b555c519921efa496",
    1337:"d2b524e59acaf17294ed761e4e0a7022044a3733067aa6e186b26ffd4a0ff48b",
    2026:"bb4495cdc6cf90647f8ebc42a5bd7e9ca2fa3a096199d2391f918acc5755ed42",
    31415:"6e832d1f383eb85ac447590c0940f2b97dcc10667972888947c6e8f0fcf600f4",
    271828:"6d9401b7e73d3379eb5972c8ce2e6bb83819ecca498ff0247db7a36b4c789cb9",
}
MATCHED_STANDALONE_GATED_REFERENCE={
    42:{"core5_accuracy":0.8723175965665236,"core5_macro_f1":0.8082784918645334,"full6_accuracy":0.8109799713876967,"full6_macro_f1":0.7618791749618042},
    1337:{"core5_accuracy":0.8706008583690987,"core5_macro_f1":0.8059615547782674,"full6_accuracy":0.8167024320457797,"full6_macro_f1":0.7575448872324794},
    2026:{"core5_accuracy":0.8746781115879829,"core5_macro_f1":0.8149207964871712,"full6_accuracy":0.8165236051502146,"full6_macro_f1":0.7628040840301965},
    31415:{"core5_accuracy":0.8759656652360516,"core5_macro_f1":0.8101638159781492,"full6_accuracy":0.8201001430615165,"full6_macro_f1":0.7566666645412546},
    271828:{"core5_accuracy":0.8718884120171675,"core5_macro_f1":0.8137020421403364,"full6_accuracy":0.8100858369098712,"full6_macro_f1":0.7716691900901737},
}

EXPECTED_GATED_STEP=int(GATED_BEST_STEP_BY_SEED[RUN_SEED])
EXPECTED_FREEZE_ID=GATED_VALIDATION_FREEZE_ID_BY_SEED[RUN_SEED]
print({
    "notebook":"Pangea FacetGate First Run",
    "RUN_SEED":RUN_SEED,
    "model_id":MODEL_ID,
    "model_revision":MODEL_REVISION,
    "expected_validation_selected_gated_step":EXPECTED_GATED_STEP,
    "shared_forward_protocol_sha256":SHARED_FORWARD_PROTOCOL_SHA256,
    "locked_test_forwards":0,
})


In [ ]:
# CELL 2 — CPU-ONLY EXACT SAME-SEED PANGEA PARENT OR SIMULTANEOUS FACETGATE DECISION
# This cell intentionally performs no torch/CUDA import and no backbone forward.
import os, json, hashlib, zipfile, shutil, re
from pathlib import Path

INPUT_ROOT=Path("/kaggle/input")
WORK_ROOT=Path("/kaggle/working")
WORK_ROOT.mkdir(parents=True,exist_ok=True)
EXPLICIT_OVERRIDE=os.environ.get("M3OFFEND_GATED_SOURCE_OVERRIDE","").strip()

PROGRESS_OVERRIDE=os.environ.get("M3OFFEND_FACETGATE_PROGRESS_OVERRIDE","").strip()

PROGRESS_SOURCE=None

def _sha256_file_cpu(path,chunk=8<<20):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()

def _content_parent_fingerprint_dir(checkpoint_dir):
    """Exact same content identity used by the Pangea frozen-parent child."""
    d=Path(checkpoint_dir)
    heads=d/"classifier_heads.pt"
    cfg=d/"adapter"/"adapter_config.json"
    aw=next((d/"adapter"/x for x in ("adapter_model.safetensors","adapter_model.bin")
             if (d/"adapter"/x).is_file()),None)
    if not (heads.is_file() and cfg.is_file() and aw is not None):
        raise FileNotFoundError(f"Incomplete checkpoint for content fingerprint: {d}")
    return hashlib.sha256(
        (_sha256_file_cpu(heads)+"|"+_sha256_file_cpu(aw)+"|"+_sha256_file_cpu(cfg)).encode()
    ).hexdigest()

def _sha256_zip_member(zf,name,chunk=8<<20):
    h=hashlib.sha256()
    with zf.open(name,"r") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def _content_parent_fingerprint_zip(zf,prefix,names):
    heads=prefix+"classifier_heads.pt"
    cfg=prefix+"adapter/adapter_config.json"
    aw=next((prefix+"adapter/"+x for x in ("adapter_model.safetensors","adapter_model.bin")
             if prefix+"adapter/"+x in names),None)
    if not (heads in names and cfg in names and aw is not None):
        raise FileNotFoundError(f"Incomplete ZIP checkpoint at {prefix}")
    return hashlib.sha256(
        (_sha256_zip_member(zf,heads)+"|"+_sha256_zip_member(zf,aw)+"|"+_sha256_zip_member(zf,cfg)).encode()
    ).hexdigest()

# Fingerprint smoke test: directory and ZIP representations of the same checkpoint
# MUST resolve to one identical byte-content identity. This prevents the old
# manifest-hash vs feature-shard-hash mismatch from recurring.
_fp_smoke_root=WORK_ROOT/f"_pangea_parent_fp_smoke_seed{RUN_SEED}"
try:
    if _fp_smoke_root.exists(): shutil.rmtree(_fp_smoke_root)
    _cp=_fp_smoke_root/"checkpoint"
    (_cp/"adapter").mkdir(parents=True,exist_ok=True)
    (_cp/"classifier_heads.pt").write_bytes(b"facetgate-controller-smoke-heads")
    (_cp/"adapter"/"adapter_model.safetensors").write_bytes(b"facetgate-controller-smoke-adapter")
    (_cp/"adapter"/"adapter_config.json").write_text('{"smoke":true}',encoding="utf-8")
    _fp_dir=_content_parent_fingerprint_dir(_cp)
    _fp_zip_path=_fp_smoke_root/"checkpoint.zip"
    with zipfile.ZipFile(_fp_zip_path,"w",compression=zipfile.ZIP_STORED) as _zf:
        _zf.write(_cp/"classifier_heads.pt","checkpoint/classifier_heads.pt")
        _zf.write(_cp/"adapter"/"adapter_model.safetensors","checkpoint/adapter/adapter_model.safetensors")
        _zf.write(_cp/"adapter"/"adapter_config.json","checkpoint/adapter/adapter_config.json")
    with zipfile.ZipFile(_fp_zip_path,"r") as _zf:
        _names=_zf.namelist()
        _fp_zip=_content_parent_fingerprint_zip(_zf,"checkpoint/",_names)
    if _fp_dir!=_fp_zip:
        raise RuntimeError(f"Parent content fingerprint smoke mismatch dir={_fp_dir} zip={_fp_zip}")
    print("[CONTROLLER PARENT FINGERPRINT SMOKE PASS]",{"fingerprint":_fp_dir,"TEST_forwards":0},flush=True)
finally:
    if _fp_smoke_root.exists(): shutil.rmtree(_fp_smoke_root)

def _pick_single_parent(by_fp):
    if len(by_fp)!=1:
        return None
    copies=next(iter(by_fp.values()))
    copies=sorted(copies,key=lambda r:(0 if r["kind"]=="directory" else 1,len(r["source"]),r["source"]))
    return copies[0]

def _decide_route(progress_source,by_fp,conflicts,explicit_gated_override=False):
    """Pure CPU decision function. Progress is reused only when scientifically resumable."""
    if len(by_fp)>1:
        raise RuntimeError(
            "FAIL_CLOSED: multiple different exact-looking same-seed validation-selected Pangea parents were found. "
            "Remove ambiguity or set M3OFFEND_GATED_SOURCE_OVERRIDE.\n"+json.dumps(by_fp,indent=2)
        )
    parent=_pick_single_parent(by_fp)

    if progress_source is not None:
        pmode=progress_source.get("execution_mode")
        if pmode=="SIMULTANEOUS_FROM_SCRATCH":
            return {
                "parent":None,
                "execution_mode":"SIMULTANEOUS_FROM_SCRATCH",
                "active_progress_source":progress_source,
                "progress_action":"RESUME_SIMULTANEOUS_PROGRESS",
                "progress_ignore_reason":None,
            }
        if pmode=="REUSE_EXISTING_GATED":
            expected_fp=progress_source.get("parent_fingerprint")
            if parent is not None and expected_fp is not None and parent.get("fingerprint")==expected_fp:
                return {
                    "parent":parent,
                    "execution_mode":"REUSE_EXISTING_GATED",
                    "active_progress_source":progress_source,
                    "progress_action":"RESUME_REUSE_PROGRESS",
                    "progress_ignore_reason":None,
                }
            # Progress cannot be mixed into another scientific trajectory.
            # Fall through exactly as requested: check Gated; if valid, fresh reuse; otherwise simultaneous from scratch.
            if parent is not None:
                reason=(
                    "progress_parent_fingerprint_missing"
                    if expected_fp is None else
                    f"progress_parent_fingerprint_mismatch progress={expected_fp} parent={parent.get('fingerprint')}"
                )
                return {
                    "parent":parent,
                    "execution_mode":"REUSE_EXISTING_GATED",
                    "active_progress_source":None,
                    "progress_action":"IGNORE_INCOMPATIBLE_PROGRESS_START_FRESH_REUSE",
                    "progress_ignore_reason":reason,
                }
            if explicit_gated_override:
                raise RuntimeError(
                    "Explicit M3OFFEND_GATED_SOURCE_OVERRIDE did not resolve to one valid frozen Gated parent. "
                    f"conflicts={json.dumps(conflicts[:50],indent=2)}"
                )
            return {
                "parent":None,
                "execution_mode":"SIMULTANEOUS_FROM_SCRATCH",
                "active_progress_source":None,
                "progress_action":"IGNORE_UNRESUMABLE_REUSE_PROGRESS_START_SIMULTANEOUS",
                "progress_ignore_reason":"matching_frozen_gated_parent_not_available",
            }
        raise RuntimeError(f"Unknown progress execution mode: {pmode!r}")

    if parent is not None:
        return {
            "parent":parent,
            "execution_mode":"REUSE_EXISTING_GATED",
            "active_progress_source":None,
            "progress_action":"NO_PROGRESS_START_FRESH_REUSE",
            "progress_ignore_reason":None,
        }
    if explicit_gated_override:
        raise RuntimeError(
            "Explicit M3OFFEND_GATED_SOURCE_OVERRIDE did not resolve to one valid frozen Gated parent. "
            f"conflicts={json.dumps(conflicts[:50],indent=2)}"
        )
    return {
        "parent":None,
        "execution_mode":"SIMULTANEOUS_FROM_SCRATCH",
        "active_progress_source":None,
        "progress_action":"NO_PROGRESS_NO_GATED_START_SIMULTANEOUS",
        "progress_ignore_reason":None if not conflicts else "no_valid_gated_parent; invalid_candidates_ignored",
    }

# Cheap decision-table smoke test. This catches regressions in the requested fallback chain.
_smoke_parent={"kind":"directory","source":"/tmp/gated","thresholds":"/tmp/t.json","fingerprint":"FP"}
_smoke_reuse={"kind":"directory","source":"/tmp/progress","execution_mode":"REUSE_EXISTING_GATED","parent_fingerprint":"FP","score":100}
_smoke_bad={"kind":"directory","source":"/tmp/progress","execution_mode":"REUSE_EXISTING_GATED","parent_fingerprint":"OTHER","score":100}
_smoke_sim={"kind":"directory","source":"/tmp/progress","execution_mode":"SIMULTANEOUS_FROM_SCRATCH","score":50001}
_smoke_cases=[
    (_smoke_reuse,{"FP":[_smoke_parent]},"REUSE_EXISTING_GATED","RESUME_REUSE_PROGRESS"),
    (_smoke_reuse,{},"SIMULTANEOUS_FROM_SCRATCH","IGNORE_UNRESUMABLE_REUSE_PROGRESS_START_SIMULTANEOUS"),
    (_smoke_bad,{"FP":[_smoke_parent]},"REUSE_EXISTING_GATED","IGNORE_INCOMPATIBLE_PROGRESS_START_FRESH_REUSE"),
    (None,{"FP":[_smoke_parent]},"REUSE_EXISTING_GATED","NO_PROGRESS_START_FRESH_REUSE"),
    (None,{},"SIMULTANEOUS_FROM_SCRATCH","NO_PROGRESS_NO_GATED_START_SIMULTANEOUS"),
    (_smoke_sim,{},"SIMULTANEOUS_FROM_SCRATCH","RESUME_SIMULTANEOUS_PROGRESS"),
]
for _p,_b,_mode,_action in _smoke_cases:
    _r=_decide_route(_p,_b,[],False)
    if _r["execution_mode"]!=_mode or _r["progress_action"]!=_action:
        raise RuntimeError(f"Controller decision smoke failed: {_r} expected={_mode}/{_action}")
print("[CONTROLLER ROUTE DECISION SMOKE PASS]",{"cases":len(_smoke_cases),"TEST_forwards":0},flush=True)

def _progress_json_from_zip(zf, member):
    try:
        return json.loads(zf.read(member).decode("utf-8"))
    except Exception:
        return None

def _progress_candidate_from_zip(zp):
    """CPU-only inspection. Never loads torch tensors."""
    try:
        with zipfile.ZipFile(zp) as zf:
            names=zf.namelist()
            records=[]
            # Strongest: simultaneous runtime manifest.
            for n in names:
                if n.endswith("runtime_guard_resume/manifest.json") or n.endswith("/manifest.json"):
                    obj=_progress_json_from_zip(zf,n)
                    if not isinstance(obj,dict): continue
                    if obj.get("execution_mode")!="SIMULTANEOUS_FROM_SCRATCH": continue
                    if int(obj.get("seed",-1))!=RUN_SEED: continue
                    if obj.get("protocol_sha256")!=SHARED_FORWARD_PROTOCOL_SHA256: continue
                    if obj.get("model_id")!=MODEL_ID or obj.get("model_revision")!=MODEL_REVISION: continue
                    if int(obj.get("test_model_forwards",-1))!=0: continue
                    step=int(obj.get("global_step",0) or 0)
                    records.append({"kind":"zip","source":str(zp),"execution_mode":"SIMULTANEOUS_FROM_SCRATCH",
                                    "score":50000+step,"evidence":n,"global_step":step})
            # Reuse-parent runtime marker / feature progress.
            reuse_score=0; reuse_fp=None; evidence=[]
            for n in names:
                if n.endswith("RUNTIME_GUARD_STOP.json"):
                    obj=_progress_json_from_zip(zf,n)
                    if not isinstance(obj,dict): continue
                    if int(obj.get("seed",-1))!=RUN_SEED: continue
                    mode=obj.get("execution_mode")
                    if mode=="REUSE_EXISTING_GATED":
                        reuse_score=max(reuse_score,40000+int(obj.get("global_step",0) or 0))
                        evidence.append(n)
                bn=Path(n).name
                m=re.fullmatch(r"facet_features_rank([01])\.json",bn)
                pm=re.fullmatch(r"facet_features_rank([01])_PARTIAL\.json",bn)
                if not (m or pm): continue
                rank=int((m or pm).group(1))
                # Require the paired NPZ in the same archive; JSON-only markers are not resumable.
                npz_member=str(Path(n).with_suffix(".npz")).replace("\\","/")
                if npz_member not in names: continue
                obj=_progress_json_from_zip(zf,n)
                if not isinstance(obj,dict): continue
                if int(obj.get("seed",-1))!=RUN_SEED: continue
                if int(obj.get("rank",-1))!=rank or int(obj.get("world",-1))!=2: continue
                if obj.get("protocol_sha256")!=SHARED_FORWARD_PROTOCOL_SHA256: continue
                if obj.get("model_id")!=MODEL_ID or obj.get("model_revision")!=MODEL_REVISION: continue
                if int(obj.get("test_model_forwards",-1))!=0: continue
                fp=obj.get("parent_checkpoint_fingerprint")
                if reuse_fp is None: reuse_fp=fp
                elif fp!=reuse_fp: 
                    raise RuntimeError(f"Progress ZIP contains conflicting frozen-parent fingerprints: {zp}")
                if m:
                    rows=int(obj.get("rows",-1))
                    if rows==4193: reuse_score+=10000
                else:
                    rows=int(obj.get("rows_completed",-1))
                    if 0<rows<=4193: reuse_score+=rows
                evidence.append(n)
            if reuse_score>0:
                records.append({"kind":"zip","source":str(zp),"execution_mode":"REUSE_EXISTING_GATED",
                                "score":reuse_score,"parent_fingerprint":reuse_fp,"evidence":evidence[:20]})
            if not records: return None
            return max(records,key=lambda r:int(r["score"]))
    except zipfile.BadZipFile:
        return None


# CPU-only scanner smoke test. This deliberately exercises the same ZIP code path
# before scanning Kaggle inputs, so missing runtime imports (such as `re`) fail
# immediately and cheaply rather than after a user mounts a multi-hour progress run.
_smoke_zip=WORK_ROOT/f"_facetgate_progress_scanner_smoke_seed{RUN_SEED}.zip"
_smoke_meta={
    "schema":"M3Offend_FacetGate_v1_0_1_feature_shard",
    "protocol_sha256":SHARED_FORWARD_PROTOCOL_SHA256,
    "rank":1,"world":2,"rows":4193,
    "model_id":MODEL_ID,"model_revision":MODEL_REVISION,"seed":RUN_SEED,
    "parent_checkpoint_fingerprint":"__CONTROLLER_SMOKE_PARENT__",
    "feature_binding_sha256":"__CONTROLLER_SMOKE_BINDING__",
    "representation_dim":1024,"test_model_forwards":0,
}
try:
    with zipfile.ZipFile(_smoke_zip,"w",compression=zipfile.ZIP_STORED) as _zf:
        _zf.writestr("facet_features/facet_features_rank1.json",json.dumps(_smoke_meta))
        _zf.writestr("facet_features/facet_features_rank1.npz",b"controller-smoke")
    _smoke=_progress_candidate_from_zip(_smoke_zip)
    if not (
        isinstance(_smoke,dict)
        and _smoke.get("execution_mode")=="REUSE_EXISTING_GATED"
        and _smoke.get("parent_fingerprint")=="__CONTROLLER_SMOKE_PARENT__"
    ):
        raise RuntimeError(f"FacetGate progress scanner smoke failed: {_smoke}")
    print("[CONTROLLER PROGRESS SCANNER SMOKE PASS]",{"seed":RUN_SEED,"TEST_forwards":0},flush=True)
finally:
    try: _smoke_zip.unlink()
    except FileNotFoundError: pass

def _progress_candidate_from_dir(root):
    records=[]
    root=Path(root)

    # Simultaneous safeguards are self-contained; keep each bundle independent.
    for mf in root.rglob("manifest.json"):
        try: obj=json.loads(mf.read_text(encoding="utf-8"))
        except Exception: continue
        if obj.get("execution_mode")!="SIMULTANEOUS_FROM_SCRATCH": continue
        if int(obj.get("seed",-1))!=RUN_SEED: continue
        if obj.get("protocol_sha256")!=SHARED_FORWARD_PROTOCOL_SHA256: continue
        if obj.get("model_id")!=MODEL_ID or obj.get("model_revision")!=MODEL_REVISION: continue
        if int(obj.get("test_model_forwards",-1))!=0: continue
        step=int(obj.get("global_step",0) or 0)
        bundle=mf.parent.parent if mf.parent.name=="runtime_guard_resume" else mf.parent
        records.append({
            "kind":"directory","source":str(bundle),
            "execution_mode":"SIMULTANEOUS_FROM_SCRATCH",
            "score":50000+step,"evidence":str(mf),"global_step":step
        })

    # Reuse-parent shards are grouped by their own run bundle, not by /kaggle/input.
    bundles={}
    shard_patterns=(
        "facet_features_rank0.json","facet_features_rank1.json",
        "facet_features_rank0_PARTIAL.json","facet_features_rank1_PARTIAL.json"
    )
    shard_files=[]
    for pat in shard_patterns:
        shard_files.extend(root.rglob(pat))
    for sm in shard_files:
        npz=sm.with_suffix(".npz")
        if not npz.is_file():
            continue  # JSON-only markers are not resumable.
        try: obj=json.loads(sm.read_text(encoding="utf-8"))
        except Exception: continue
        if int(obj.get("seed",-1))!=RUN_SEED: continue
        if obj.get("protocol_sha256")!=SHARED_FORWARD_PROTOCOL_SHA256: continue
        if obj.get("model_id")!=MODEL_ID or obj.get("model_revision")!=MODEL_REVISION: continue
        if int(obj.get("test_model_forwards",-1))!=0: continue

        bundle=sm.parent.parent if sm.parent.name=="facet_features" else sm.parent
        key=str(bundle.resolve())
        rec=bundles.setdefault(key,{
            "kind":"directory","source":str(bundle),
            "execution_mode":"REUSE_EXISTING_GATED",
            "score":0,"parent_fingerprint":None,"evidence":[]
        })
        fp=obj.get("parent_checkpoint_fingerprint")
        if rec["parent_fingerprint"] is None:
            rec["parent_fingerprint"]=fp
        elif fp!=rec["parent_fingerprint"]:
            raise RuntimeError(f"Progress directory contains conflicting frozen-parent fingerprints: {bundle}")

        if sm.name.endswith("_PARTIAL.json"):
            rows=int(obj.get("rows_completed",-1))
            if 0<rows<=4193: rec["score"]+=rows
        else:
            rows=int(obj.get("rows",-1))
            if rows==4193: rec["score"]+=10000
        rec["evidence"].append(str(sm))

    records.extend(r for r in bundles.values() if int(r["score"])>0)
    return max(records,key=lambda r:int(r["score"])) if records else None

_progress_candidates=[]
if PROGRESS_OVERRIDE:
    _pp=Path(PROGRESS_OVERRIDE)
    if not _pp.exists():
        raise FileNotFoundError(f"M3OFFEND_FACETGATE_PROGRESS_OVERRIDE does not exist: {_pp}")
    _pr=_progress_candidate_from_zip(_pp) if _pp.is_file() and zipfile.is_zipfile(_pp) else _progress_candidate_from_dir(_pp)
    if _pr is None:
        raise RuntimeError(f"Explicit FacetGate progress source has no valid same-seed resumable state: {_pp}")
    _progress_candidates=[_pr]
else:
    for _zp in INPUT_ROOT.rglob("*.zip"):
        _pr=_progress_candidate_from_zip(_zp)
        if _pr is not None: _progress_candidates.append(_pr)
    # Also allow an already-extracted dataset directory.
    _pr=_progress_candidate_from_dir(INPUT_ROOT)
    if _pr is not None: _progress_candidates.append(_pr)

if _progress_candidates:
    _progress_candidates=sorted(_progress_candidates,key=lambda r:(-int(r["score"]),len(r["source"]),r["source"]))
    top_score=int(_progress_candidates[0]["score"])
    tied=[r for r in _progress_candidates if int(r["score"])==top_score]
    modes={r["execution_mode"] for r in tied}
    fps={r.get("parent_fingerprint") for r in tied if r.get("parent_fingerprint") is not None}
    if len(modes)>1 or len(fps)>1:
        raise RuntimeError("FAIL_CLOSED: ambiguous same-seed FacetGate progress inputs at equal best progress.\n"+
                           json.dumps(tied,indent=2))
    PROGRESS_SOURCE=_progress_candidates[0]
    print("[FACETGATE PROGRESS FOUND]",json.dumps(PROGRESS_SOURCE,indent=2))
else:
    print("[FACETGATE PROGRESS NOT FOUND] Next: check exact same-seed Gated parent; if absent, start simultaneous Gated+FacetGate.",flush=True)

def _read_json_bytes(b):
    return json.loads(b.decode("utf-8"))

def _freeze_ok(obj):
    if not isinstance(obj,dict): return False,"freeze_not_dict"
    schema=str(obj.get("schema",""))
    if "Pangea" not in schema or "ValidationFreeze" not in schema:
        return False,f"unexpected_freeze_schema={schema}"
    try:
        seed=int(obj.get("seed",-1)); step=int(obj.get("best_step",-1)); tf=int(obj.get("test_forward_count",-1))
    except Exception:
        return False,"freeze_scalar_parse_failure"
    if seed!=RUN_SEED: return False,f"seed={seed}"
    if step!=EXPECTED_GATED_STEP: return False,f"best_step={step}"
    if obj.get("common_scientific_hash")!=PARENT_GATED_PROTOCOL_SHA256:
        return False,"parent_protocol_mismatch"
    if tf!=0 or not bool(obj.get("test_locked",False)):
        return False,f"test_state=locked:{obj.get('test_locked')} forwards:{obj.get('test_forward_count')}"
    if obj.get("validation_freeze_id")!=EXPECTED_FREEZE_ID:
        return False,f"validation_freeze_id={obj.get('validation_freeze_id')}"
    return True,"ok"

def _manifest_ok(obj):
    if not isinstance(obj,dict): return False,"manifest_not_dict"
    try:
        seed=int(obj.get("seed",-1)); step=int(obj.get("global_step",-1))
    except Exception:
        return False,"manifest_scalar_parse_failure"
    if seed!=RUN_SEED: return False,f"seed={seed}"
    if step!=EXPECTED_GATED_STEP: return False,f"global_step={step}"
    if obj.get("model_id")!=MODEL_ID: return False,f"model_id={obj.get('model_id')}"
    if obj.get("model_revision")!=MODEL_REVISION: return False,f"model_revision={obj.get('model_revision')}"
    if obj.get("frozen_common_gated_protocol_sha256")!=PARENT_GATED_PROTOCOL_SHA256:
        return False,"parent_protocol_mismatch"
    if obj.get("pangea_native_binding_sha256")!=PANGEA_NATIVE_BINDING_SHA256:
        return False,f"native_binding={obj.get('pangea_native_binding_sha256')}"
    return True,"ok"

def _direct_threshold(root):
    candidates=[]
    for rel in ["artifacts/thresholds.json","thresholds.json","artifacts/validation_thresholds.json","validation_thresholds.json"]:
        p=root/rel
        if p.is_file(): candidates.append(p)
    if not candidates:
        for name in ["thresholds.json","validation_thresholds.json"]:
            candidates.extend(root.rglob(name))
    return sorted(set(candidates),key=lambda p:(len(str(p)),str(p)))[0] if candidates else None

def _direct_candidates(search_root):
    rows=[]; conflicts=[]
    if not search_root.exists(): return rows,conflicts
    freezes=list(search_root.rglob("STRICT_VALIDATION_FREEZE.json"))
    for fp in freezes:
        try: fr=json.loads(fp.read_text(encoding="utf-8"))
        except Exception as e:
            conflicts.append({"kind":"directory","source":str(fp),"reason":f"freeze_unreadable:{type(e).__name__}"})
            continue
        # Other publication seeds are unrelated, not conflicting intended parents.
        try: fr_seed=int(fr.get("seed",-1))
        except Exception: fr_seed=-1
        if fr_seed!=RUN_SEED:
            continue
        ok,reason=_freeze_ok(fr)
        root=fp.parent.parent if fp.parent.name=="artifacts" else fp.parent
        if not ok:
            conflicts.append({"kind":"directory","source":str(root),"reason":reason}); continue
        cp_options=[root/"checkpoints"/f"checkpoint-{EXPECTED_GATED_STEP:04d}",root/"checkpoints"/f"checkpoint-{EXPECTED_GATED_STEP}"]
        cp=next((p for p in cp_options if p.is_dir()),None)
        if cp is None:
            # allow one nesting layer used when mounted Kaggle outputs retain an outer folder
            hits=[p for p in root.rglob(f"checkpoint-{EXPECTED_GATED_STEP:04d}") if p.is_dir()]
            hits += [p for p in root.rglob(f"checkpoint-{EXPECTED_GATED_STEP}") if p.is_dir()]
            cp=sorted(set(hits),key=lambda p:(len(str(p)),str(p)))[0] if hits else None
        if cp is None:
            conflicts.append({"kind":"directory","source":str(root),"reason":"selected_checkpoint_missing"}); continue
        mf=cp/"checkpoint_manifest.json"
        heads=cp/"classifier_heads.pt"; ac=cp/"adapter"/"adapter_config.json"
        aw=next((cp/"adapter"/x for x in ["adapter_model.safetensors","adapter_model.bin"] if (cp/"adapter"/x).is_file()),None)
        if not (mf.is_file() and heads.is_file() and ac.is_file() and aw is not None):
            conflicts.append({"kind":"directory","source":str(cp),"reason":"selected_checkpoint_incomplete"}); continue
        try: mo=json.loads(mf.read_text(encoding="utf-8"))
        except Exception as e:
            conflicts.append({"kind":"directory","source":str(cp),"reason":f"manifest_unreadable:{type(e).__name__}"}); continue
        ok,reason=_manifest_ok(mo)
        if not ok:
            conflicts.append({"kind":"directory","source":str(cp),"reason":reason}); continue
        th=_direct_threshold(root)
        if th is None:
            conflicts.append({"kind":"directory","source":str(root),"reason":"validation_thresholds_missing"}); continue
        # Use the SAME content fingerprint as the frozen-parent child and feature shards.
        fingerprint=_content_parent_fingerprint_dir(cp)
        rows.append({"kind":"directory","source":str(cp),"thresholds":str(th),"fingerprint":fingerprint,
                     "manifest_checkpoint_content_sha256":mo.get("checkpoint_content_sha256"),
                     "validation_freeze":str(fp),"checkpoint_manifest":str(mf)})
    return rows,conflicts

def _zip_candidates(search_root):
    rows=[]; conflicts=[]
    if not search_root.exists(): return rows,conflicts
    for zp in search_root.rglob("*.zip"):
        # Avoid opening arbitrary giant unrelated archives unless Pangea/Gated naming or required members prove relevance.
        try:
            with zipfile.ZipFile(zp) as zf:
                names=zf.namelist()
                freeze_members=[n for n in names if n.endswith("artifacts/STRICT_VALIDATION_FREEZE.json") or n.endswith("STRICT_VALIDATION_FREEZE.json")]
                if not freeze_members:
                    continue
                for fm in freeze_members:
                    try: fr=_read_json_bytes(zf.read(fm))
                    except Exception as e:
                        conflicts.append({"kind":"zip","source":str(zp),"reason":f"freeze_unreadable:{type(e).__name__}"}); continue
                    try: fr_seed=int(fr.get("seed",-1))
                    except Exception: fr_seed=-1
                    if fr_seed!=RUN_SEED:
                        continue
                    ok,reason=_freeze_ok(fr)
                    if not ok:
                        conflicts.append({"kind":"zip","source":str(zp),"reason":reason}); continue

                    tags=[f"checkpoint-{EXPECTED_GATED_STEP:04d}/checkpoint_manifest.json",f"checkpoint-{EXPECTED_GATED_STEP}/checkpoint_manifest.json"]
                    mm=[n for n in names if any(n.endswith(t) for t in tags)]
                    if len(mm)!=1:
                        conflicts.append({"kind":"zip","source":str(zp),"reason":f"selected_checkpoint_manifest_count={len(mm)}"}); continue
                    manifest_member=mm[0]
                    prefix=manifest_member[:-len("checkpoint_manifest.json")]
                    try: mo=_read_json_bytes(zf.read(manifest_member))
                    except Exception as e:
                        conflicts.append({"kind":"zip","source":str(zp),"reason":f"manifest_unreadable:{type(e).__name__}"}); continue
                    ok,reason=_manifest_ok(mo)
                    if not ok:
                        conflicts.append({"kind":"zip","source":str(zp),"reason":reason}); continue
                    required=[prefix+"classifier_heads.pt",prefix+"adapter/adapter_config.json"]
                    adapter_members=[prefix+"adapter/adapter_model.safetensors",prefix+"adapter/adapter_model.bin"]
                    if not all(x in names for x in required) or not any(x in names for x in adapter_members):
                        conflicts.append({"kind":"zip","source":str(zp),"reason":"selected_checkpoint_incomplete"}); continue
                    tm=[n for n in names if n.endswith("artifacts/thresholds.json") or n.endswith("/thresholds.json")]
                    if not tm:
                        conflicts.append({"kind":"zip","source":str(zp),"reason":"validation_thresholds_missing"}); continue
                    tm=sorted(tm,key=len)[0]
                    out=WORK_ROOT/f"_pangea_facetgate_seed{RUN_SEED}_{hashlib.sha256(str(zp).encode()).hexdigest()[:10]}_thresholds.json"
                    out.write_bytes(zf.read(tm))
                    # Same byte-content fingerprint as direct checkpoints and frozen-parent feature shards.
                    fingerprint=_content_parent_fingerprint_zip(zf,prefix,names)
                    rows.append({"kind":"zip","source":str(zp),"thresholds":str(out),"fingerprint":fingerprint,
                                 "manifest_checkpoint_content_sha256":mo.get("checkpoint_content_sha256"),
                                 "validation_freeze_member":fm,"checkpoint_manifest_member":manifest_member})
        except zipfile.BadZipFile:
            continue
        except Exception as e:
            # Only treat it as a conflict if the file name itself strongly signals this exact seed Pangea Gated output.
            low=zp.name.casefold()
            if ("pangea" in low and "gated" in low and str(RUN_SEED) in low):
                conflicts.append({"kind":"zip","source":str(zp),"reason":f"zip_scan_error:{type(e).__name__}:{e}"})
    return rows,conflicts

if EXPLICIT_OVERRIDE:
    roots=[Path(EXPLICIT_OVERRIDE)]
    if not roots[0].exists():
        raise FileNotFoundError(f"M3OFFEND_GATED_SOURCE_OVERRIDE does not exist: {roots[0]}")
else:
    roots=[INPUT_ROOT]

valid=[]; conflicts=[]
for root in roots:
    if root.is_file() and zipfile.is_zipfile(root):
        # Scan the parent but keep only the explicit file.
        rr,cc=_zip_candidates(root.parent)
        rr=[x for x in rr if Path(x["source"]).resolve()==root.resolve()]
        cc=[x for x in cc if Path(x["source"]).resolve()==root.resolve()]
    elif root.is_dir():
        rr,cc=_direct_candidates(root)
        rz,cz=_zip_candidates(root)
        rr+=rz; cc+=cz
    else:
        rr=[]; cc=[{"kind":"override","source":str(root),"reason":"unsupported_override"}]
    valid.extend(rr); conflicts.extend(cc)

# Collapse copies of the exact same validation-selected checkpoint by the
# exact byte-content fingerprint used inside the Pangea child.
by_fp={}
for r in valid:
    by_fp.setdefault(r["fingerprint"],[]).append(r)

_route=_decide_route(
    PROGRESS_SOURCE,by_fp,conflicts,
    explicit_gated_override=bool(EXPLICIT_OVERRIDE)
)
PARENT=_route["parent"]
EXECUTION_MODE=_route["execution_mode"]
ACTIVE_PROGRESS_SOURCE=_route["active_progress_source"]
PROGRESS_ACTION=_route["progress_action"]
PROGRESS_IGNORE_REASON=_route["progress_ignore_reason"]

if PROGRESS_SOURCE is not None and ACTIVE_PROGRESS_SOURCE is None:
    print("[FACETGATE PROGRESS NOT USED]",{
        "action":PROGRESS_ACTION,
        "reason":PROGRESS_IGNORE_REASON,
        "discovered_progress_source":PROGRESS_SOURCE.get("source"),
        "next_execution_mode":EXECUTION_MODE,
        "TEST_forwards":0,
    },flush=True)
elif ACTIVE_PROGRESS_SOURCE is not None:
    print("[FACETGATE PROGRESS ACTIVE]",{
        "action":PROGRESS_ACTION,
        "source":ACTIVE_PROGRESS_SOURCE.get("source"),
        "execution_mode":EXECUTION_MODE,
        "TEST_forwards":0,
    },flush=True)

controller_record={
    "schema":"M3Offend_Pangea7BHF_FacetGate_AutoSeed_Controller_v1_0_1",
    "seed":RUN_SEED,
    "model_id":MODEL_ID,"model_revision":MODEL_REVISION,
    "expected_validation_selected_gated_step":EXPECTED_GATED_STEP,
    "expected_validation_freeze_id":EXPECTED_FREEZE_ID,
    "execution_mode":EXECUTION_MODE,
    "progress_source_discovered":PROGRESS_SOURCE,
    "progress_source_active":ACTIVE_PROGRESS_SOURCE,
    "progress_action":PROGRESS_ACTION,
    "progress_ignore_reason":PROGRESS_IGNORE_REASON,
    "parent":PARENT,
    "valid_parent_state_count":len(by_fp),
    "same_seed_conflicts":conflicts,
    "shared_forward_protocol_sha256":SHARED_FORWARD_PROTOCOL_SHA256,
    "locked_test_model_forwards":0,
}
controller_path=WORK_ROOT/f"M3Offend_Pangea7BHF_FacetGate_Controller_Seed{RUN_SEED}.json"
controller_path.write_text(json.dumps(controller_record,indent=2),encoding="utf-8")
print("[FACETGATE CONTROLLER DECISION]",json.dumps(controller_record,indent=2))

In [ ]:
# CELL 3 — EMBEDDED REVIEWED PANGEA FACETGATE CHILD PIPELINES (compressed)
# Scientific protocol is unchanged. Runtime fixes add progress-resume orchestration
# and remove unnecessary NCCL collectives from frozen-parent feature export.
SIM_CHILD_ZLIB_B64 = '...'
SIM_TEMPLATE_SHA256 = 'a633430c791f88ff5a268a4d82ec87fe40241f7238a2a9f4ece8146f22a9bc29'
REUSE_CHILD_ZLIB_B64 = '...'
REUSE_TEMPLATE_SHA256 = '0f4e3a79d4575da1c5adb1a506f868f86b7cf2be08db759e182684e136c4c8ba'

print({
    "simultaneous_template_sha256":SIM_TEMPLATE_SHA256,
    "reuse_parent_template_sha256":REUSE_TEMPLATE_SHA256,
    "shared_forward_protocol_sha256":SHARED_FORWARD_PROTOCOL_SHA256,
})


In [ ]:
# CELL 4 — MATERIALIZE SELECTED FACETGATE PIPELINE WITH EXACT SEED/PARENT BINDING
import base64, zlib, json, hashlib
from pathlib import Path

def _decode_template(blob,expected_sha):
    raw=zlib.decompress(base64.b64decode(blob.encode("ascii")))
    observed=hashlib.sha256(raw).hexdigest()
    if observed!=expected_sha:
        raise RuntimeError(f"Embedded child template SHA mismatch expected={expected_sha} observed={observed}")
    return json.loads(raw.decode("utf-8"))

if EXECUTION_MODE=="REUSE_EXISTING_GATED":
    child=_decode_template(REUSE_CHILD_ZLIB_B64,REUSE_TEMPLATE_SHA256)
else:
    child=_decode_template(SIM_CHILD_ZLIB_B64,SIM_TEMPLATE_SHA256)

for cell in child.get("cells",[]):
    if cell.get("cell_type")!="code":
        continue
    src="".join(cell.get("source",[]))
    src=src.replace("__RUN_SEED__",str(int(RUN_SEED)))
    src=src.replace('FACETGATE_PROGRESS_SOURCE_OVERRIDE=""',
                    f'FACETGATE_PROGRESS_SOURCE_OVERRIDE={((ACTIVE_PROGRESS_SOURCE or {}).get("source") or "")!r}')
    if EXECUTION_MODE=="REUSE_EXISTING_GATED":
        src=src.replace('GATED_SOURCE_OVERRIDE=""',f'GATED_SOURCE_OVERRIDE={PARENT["source"]!r}')
        src=src.replace('GATED_THRESHOLDS_OVERRIDE=""',f'GATED_THRESHOLDS_OVERRIDE={PARENT["thresholds"]!r}')
    cell["source"]=src.splitlines(keepends=True)
    cell["execution_count"]=None
    cell["outputs"]=[]

child_name=(
    f"M3Offend_9318_Pangea7BHF_FacetGate_Seed{RUN_SEED}_REUSE_EXISTING_PARENT.ipynb"
    if EXECUTION_MODE=="REUSE_EXISTING_GATED"
    else f"M3Offend_9318_Pangea7BHF_FacetGate_Seed{RUN_SEED}_SIMULTANEOUS.ipynb"
)
CHILD_NOTEBOOK=WORK_ROOT/child_name
CHILD_NOTEBOOK.write_text(json.dumps(child,ensure_ascii=False,indent=1),encoding="utf-8")

# Implementation-level fail-closed QA for the corrected Pangea runtime binding.
_child_text="\n".join("".join(c.get("source",[])) for c in child.get("cells",[]) if c.get("cell_type")=="code")
if EXECUTION_MODE=="REUSE_EXISTING_GATED":
    required_tokens=[
        "ResidualAdapter.norm+net",
        "Discriminative state mismatch",
        "child.load_state_dict(sub,strict=True)",
        'torch.set_float32_matmul_precision("high")',
        "EXPECTED_PANGEA_GATED_DISCRIMINATIVE_KEYS",
        "adapters.{task}.net.0.weight",
        "TRAINING_SOFT_STOP_SECONDS=11*3600",
        "RUNTIME_GUARD_STOP",
        "independent_per_gpu_no_nccl_collectives",
        "periodic_progress_checkpoint",
        "Current_Progress.zip",
    ]
    # The human-readable interface spec carries the adapter-layout token.
    missing=[x for x in required_tokens if x not in _child_text]
    if missing: raise RuntimeError(f"Corrected Pangea reuse invariants missing: {missing}")
    if "model.load_state_dict(state,strict=False)" in _child_text:
        raise RuntimeError("Forbidden whole-wrapper discriminative load detected in reuse child")
else:
    required_tokens=[
        'order_gen.manual_seed(SEED+100000*int(epoch))',
        'torch.set_float32_matmul_precision("high")',
        'foreach=False,fused=False',
        'self.net=nn.Sequential',
        'activation_saved_tensor_context',
        'optimizer_moments_to_cpu',
        'optimizer_moments_to_param_devices',
        'random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)',
        'save_runtime_resume',
        '11h00_training_soft_stop',
        'progress_source',
        'sync_mutable_state_before_ddp',
        'set_runtime_activation_mode(0)',
        'activation_mode_backoff_after_oom',
        'exact_manual_gradient_average',
        'shared_forward_memory_smoke',
        'PYTORCH_CUDA_ALLOC_CONF',
        'LAUNCHER HOST-RAM CLEANUP',
    ]
    missing=[x for x in required_tokens if x not in _child_text]
    if missing: raise RuntimeError(f"Corrected Pangea simultaneous invariants missing: {missing}")

# Static Python compilation before any child execution.
for i,cell in enumerate(child["cells"]):
    if cell.get("cell_type")=="code":
        compile("".join(cell.get("source",[])),f"{child_name}:cell{i}","exec")

CHILD_SHA256=hashlib.sha256(CHILD_NOTEBOOK.read_bytes()).hexdigest()
print("[FACETGATE CHILD STATIC QA PASS]",{
    "mode":EXECUTION_MODE,
    "seed":RUN_SEED,
    "path":str(CHILD_NOTEBOOK),
    "sha256":CHILD_SHA256,
    "protocol_sha256":SHARED_FORWARD_PROTOCOL_SHA256,
    "locked_TEST_model_forwards":0,
})


In [ ]:
# CELL 5 — EXECUTE THE SELECTED FACETGATE PIPELINE IN A FRESH KERNEL
# Controller remains CPU-only; the selected child owns the GPU / torchrun lifecycle.
import subprocess, sys, time, os, signal, threading
from pathlib import Path

EXECUTED_NOTEBOOK=CHILD_NOTEBOOK.with_name(CHILD_NOTEBOOK.stem+"_EXECUTED.ipynb")
cmd=[
    sys.executable,"-m","jupyter","nbconvert",
    "--to","notebook","--execute",
    "--ExecutePreprocessor.timeout=-1",
    "--output",EXECUTED_NOTEBOOK.name,
    str(CHILD_NOTEBOOK),
]
print("[EXECUTING FACETGATE]"," ".join(cmd),flush=True)
started=time.time()
proc=subprocess.Popen(cmd,cwd=str(WORK_ROOT),start_new_session=True)

# 11:25 graceful watchdog for either execution family. The worker also has its
# own elapsed-time soft stop; this file is a second independent stop signal.
if EXECUTION_MODE=="REUSE_EXISTING_GATED":
    _guard_root=Path(f"/kaggle/working/M3Offend_9318_FacetGate_Pangea7BHF_Seed{RUN_SEED}")
else:
    _guard_root=Path(f"/kaggle/working/M3Offend_9318_FacetGate_Pangea7BHF_Seed{RUN_SEED}_SIMULTANEOUS")
_guard_stop_request=_guard_root/"STOP_REQUEST"

def _controller_graceful_watchdog():
    watchdog_at=CONTROLLER_SESSION_START_UNIX+(11*3600+25*60)
    time.sleep(max(0.0,watchdog_at-time.time()))
    try:
        _guard_root.mkdir(parents=True,exist_ok=True)
        if not _guard_stop_request.exists():
            _guard_stop_request.write_text("11h25 controller watchdog requested graceful FacetGate stop\n",encoding="utf-8")
            print("[CONTROLLER WATCHDOG] 11h25 graceful STOP_REQUEST written.",flush=True)
    except Exception as exc:
        print("[CONTROLLER WATCHDOG] Could not write graceful STOP_REQUEST:",repr(exc),flush=True)

threading.Thread(target=_controller_graceful_watchdog,daemon=True).start()

# Controller-level emergency ceiling. The child/worker owns the scientific
# 11:00/11:25 graceful safeguard; these signals are only last-resort protection
# against Kaggle's 12-hour hard termination.
def _controller_emergency_watchdog():
    term_at=CONTROLLER_SESSION_START_UNIX+(11*3600+50*60)
    time.sleep(max(0.0,term_at-time.time()))
    if proc.poll() is None:
        print("[CONTROLLER WATCHDOG] 11h50 child still alive; sending SIGTERM.",flush=True)
        try: os.killpg(proc.pid,signal.SIGTERM)
        except ProcessLookupError: pass
    kill_at=CONTROLLER_SESSION_START_UNIX+(11*3600+55*60)
    time.sleep(max(0.0,kill_at-time.time()))
    if proc.poll() is None:
        print("[CONTROLLER WATCHDOG] 11h55 child still alive; sending SIGKILL.",flush=True)
        try: os.killpg(proc.pid,signal.SIGKILL)
        except ProcessLookupError: pass

threading.Thread(target=_controller_emergency_watchdog,daemon=True).start()
rc=proc.wait()
elapsed=time.time()-started
if rc!=0:
    raise RuntimeError(
        f"FacetGate child failed mode={EXECUTION_MODE} returncode={rc}. "
        f"Inspect {EXECUTED_NOTEBOOK if EXECUTED_NOTEBOOK.exists() else CHILD_NOTEBOOK}."
    )
print("[FACETGATE EXECUTION COMPLETE]",{
    "mode":EXECUTION_MODE,"seed":RUN_SEED,
    "elapsed_seconds":elapsed,
    "executed_notebook":str(EXECUTED_NOTEBOOK),
    "locked_TEST_model_forwards":0,
})


In [ ]:
# CELL 6 — FINAL FACETGATE DEVELOPMENT / SAFE-STOP SUMMARY
import json
from pathlib import Path

resume_safeguards=sorted(Path("/kaggle/working").glob(f"M3Offend_9318_Pangea7BHF_FacetGate_Seed{RUN_SEED}_*Resume_Safeguard.zip"))
runtime_stop_markers=sorted(Path("/kaggle/working").rglob("RUNTIME_GUARD_STOP.json"))
if resume_safeguards or runtime_stop_markers:
    print("[FACETGATE RUNTIME SAFEGUARD STATUS]",{
        "seed":RUN_SEED,
        "resume_safeguards":[str(x) for x in resume_safeguards],
        "runtime_stop_markers":[str(x) for x in runtime_stop_markers],
        "locked_TEST_model_forwards":0,
    })

if EXECUTION_MODE=="REUSE_EXISTING_GATED":
    root=Path(f"/kaggle/working/M3Offend_9318_FacetGate_Pangea7BHF_Seed{RUN_SEED}")
    result_path=root/"facetgate_result.json"
    result=json.loads(result_path.read_text(encoding="utf-8")) if result_path.exists() else None
    zips=sorted(Path("/kaggle/working").glob(f"M3Offend_9318_FacetGate_Pangea7BHF_Seed{RUN_SEED}_RESULTS.zip"))
    print("[FINAL FACETGATE]",{
        "mode":"REUSE_EXISTING_GATED",
        "scientific_family":"frozen-parent FacetGate reproduction path",
        "parent_validation_selected_step":EXPECTED_GATED_STEP,
        "parent_validation_freeze_id":EXPECTED_FREEZE_ID,
        "result_root":str(root),
        "result_zip":[str(x) for x in zips],
        "result":result,
        "locked_TEST_model_forwards":0,
    })
else:
    root=Path(f"/kaggle/working/M3Offend_9318_FacetGate_Pangea7BHF_Seed{RUN_SEED}_SIMULTANEOUS")
    result_path=root/"artifacts"/"facetgate_result.json"
    result=json.loads(result_path.read_text(encoding="utf-8")) if result_path.exists() else None
    manifest_path=root/"artifacts"/"dual_checkpoint_final_manifest.json"
    manifest=json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else None
    zips=sorted(Path("/kaggle/working").glob(f"M3Offend_9318_FacetGate_Pangea7BHF_Seed{RUN_SEED}_SIMULTANEOUS_RESULTS.zip"))
    print("[FINAL FACETGATE]",{
        "mode":"SIMULTANEOUS_FROM_SCRATCH",
        "scientific_family":"protocol v1.0.1 simultaneous shared-forward FacetGate",
        "result_root":str(root),
        "result_zip":[str(x) for x in zips],
        "dual_checkpoint_manifest":manifest,
        "result":result,
        "locked_TEST_model_forwards":0,
    })
